# 面试问题：Self-Consistency 为什么可能提升推理？怎样处理答案解析、相关样本、加权投票和提前停止？

**一句话回答**：对同一问题采样多条具有差异的推理路径，把最终答案规范化后边缘化/投票，可降低单一路径偶然错误；但收益依赖错误不完全相关。重复模板、错误解析和同源 verifier 会制造虚假多数，因此要做路径聚类、置信拒答和质量—成本评测。

本 Notebook 不调用模型 API，而是在受控候选上实现答案规范化、多数/加权投票、相关性有效样本量、顺序停止、聚类去重和 pass@k 对照。


In [ ]:
from collections import Counter,defaultdict
import math,re
import numpy as np

SEED135=13501; rng135=np.random.default_rng(SEED135)
assert SEED135==13501
assert Counter([1,1,2]).most_common(1)[0]==(1,2)
assert np.isfinite(rng135.normal())


## 1. 先定义 final answer contract

投票对象不是整段 CoT，而是经任务特定 parser 规范化后的答案。数学题需处理逗号、小数/分数和单位，分类题只接受标签集合；解析失败单独计数，不能默认变成空字符串并抱团获胜。下面只接受显式 `Answer:` 后的整数。


In [ ]:
def parse_answer135(text):
    m=re.search(r"Answer\s*:\s*([-+]?\d+)(?![\d.])",text,re.I)
    return int(m.group(1)) if m else None
assert parse_answer135("work... Answer: 42")==42
assert parse_answer135("answer=-3") is None
assert parse_answer135("Answer: 42.5") is None


## 2. 基础 Self-Consistency 对规范答案做多数票

忽略不可解析候选后统计频数；并列时不应依赖输入顺序，需按预先定义的 verifier、平均置信或直接拒答。多数票只聚合 final answer，不证明获胜路径中的每一步都正确。


In [ ]:
def majority135(answers):
    valid=[a for a in answers if a is not None]; counts=Counter(valid)
    if not counts: return None,0.0
    top=counts.most_common(); best=top[0][1]
    winners=sorted(k for k,v in top if v==best)
    return (winners[0] if len(winners)==1 else None),best/len(valid)
ans135,conf135=majority135([42,41,42,None,42])
assert ans135==42
assert math.isclose(conf135,.75)
assert majority135([1,2])[0] is None


## 3. 加权票需要校准，不能直接用长度或自信措辞

可按外部 verifier、规则检查或校准后的 path score 加权；模型 token likelihood 常偏好短答案，未经校准会引入新偏差。权重应非负、有限并有版本。下面聚合 answer 权重，并展示少数高质量路径可以纠正低质量多数。


In [ ]:
def weighted_vote135(pairs):
    totals=defaultdict(float)
    for a,w in pairs:
        if a is not None and np.isfinite(w) and w>=0: totals[a]+=w
    ordered=sorted(totals.items(),key=lambda x:(-x[1],x[0])); return ordered[0] if ordered else (None,0.)
winner135,weight135=weighted_vote135([(41,.2),(41,.2),(42,.9)])
assert winner135==42
assert math.isclose(weight135,.9)
assert weighted_vote135([(None,1.)])[0] is None


## 4. 相关错误让名义样本数虚高

同一温度、同一开头或 beam 分支常产生高度相似路径。若平均两两相关系数为 `ρ`，粗略有效样本量可写作 `n/(1+(n-1)ρ)`；`ρ→1` 时采 100 条也接近 1 条独立证据。应改变 seed/temperature/prompt 或按推理骨架聚类。


In [ ]:
def effective_n135(n,rho): return n/(1+(n-1)*rho)
assert effective_n135(10,0)==10
assert math.isclose(effective_n135(10,1),1)
assert effective_n135(20,.8)<effective_n135(10,.1)


## 5. 先按 reasoning signature 聚类再投票

可移除数字和表面措辞后提取步骤操作序列，近似识别复制路径；每个 cluster 先产生一票或按 cluster size 次线性加权。这里用受控 `signature` 字段演示：三条复制错误不再自动压倒两条独立正确路径。


In [ ]:
paths135=[("bad-template",41),("bad-template",41),("bad-template",41),("derive-a",42),("derive-b",42)]
def cluster_vote135(paths):
    by_sig={}
    for sig,a in paths: by_sig.setdefault(sig,a)
    return majority135(list(by_sig.values()))
assert majority135([a for _,a in paths135])[0]==41
assert cluster_vote135(paths135)[0]==42
assert len({s for s,_ in paths135})==3


## 6. 顺序采样可在多数已不可逆时提前停止

若剩余预算全部投给第二名也无法追平第一名，就可确定性停止；更激进的概率停止需校准误差界。停止逻辑要把 parse failure 计入已消耗成本，并设置最小样本数，避免前两条偶然一致就过早结束。


In [ ]:
def irreversible135(answers,max_samples,min_samples=3):
    if len(answers)<min_samples: return False
    c=Counter(a for a in answers if a is not None); top=sorted(c.values(),reverse=True)+[0,0]; remaining=max_samples-len(answers)
    return top[0]>top[1]+remaining
assert not irreversible135([42,42],7)
assert irreversible135([42,42,42,42,41],6)
assert not irreversible135([42,42,41],7)


## 7. 低票差、低解析率或冲突 verifier 时应拒答

面试中要把“多采样”与“可信回答”分开。若最高票占比低、前两名接近、有效独立 cluster 少或硬规则失败，输出不确定/转工具/人工，而不是总给多数答案。阈值需在验证集按错误成本校准。


In [ ]:
def decide135(answers,threshold=.6):
    a,c=majority135(answers); return a if a is not None and c>=threshold else "ABSTAIN"
assert decide135([1,1,2])==1
assert decide135([1,2,3])=="ABSTAIN"
assert decide135([None,None])=="ABSTAIN"


## 8. 区分 pass@k、majority accuracy 与单位成本净收益

`pass@k` 只问候选中是否存在正确解，需要 oracle 才能挑中；self-consistency accuracy 问聚合后是否正确。报告单样本、不同 k/温度的 accuracy、coverage、解析率、有效样本量、token/延迟和边际收益，不能用 pass@k 冒充可部署准确率。


In [ ]:
cases135=[([1,2,2],2),([3,4,3],4),([5,6,7],7)]
passk135=np.mean([gold in xs for xs,gold in cases135]); majority_acc135=np.mean([majority135(xs)[0]==gold for xs,gold in cases135])
assert passk135==1
assert majority_acc135<passk135
assert math.isclose(majority_acc135,1/3)


## 面试总结

完整链路是：**同题多样化采样 → task-specific final parser → parse failure 单列 → 多数或校准 verifier 加权 → reasoning signature 聚类 → 估计相关性有效样本量 → 票差/预算顺序停止 → 低置信拒答 → 对比 single、pass@k 和聚合 accuracy 的成本曲线**。Self-Consistency 的核心假设是错误具有足够差异，而不是“答案越多越正确”。

延伸阅读：[Self-Consistency](https://arxiv.org/abs/2203.11171)、[Chain-of-Thought Prompting](https://arxiv.org/abs/2201.11903)、[Let's Verify Step by Step](https://arxiv.org/abs/2305.20050)。
